# Neural baselines on colour-MNIST

Loads the autoencoder and **both** neural baselines, and renders all 180
`(digit, foreground, background)` combinations for each.

The two baselines answer different questions:

- **deterministic** predicts one latent per label. It is a point predictor, so every draw
  for a combination is the same image.
- **mixture** is a conditional Gaussian mixture over the latent, so it has a real
  conditional distribution to draw from.

Both expose the same `sample(labels, std_correction)` interface a CSPN does, so nothing
below is specific to a baseline.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision.utils as vutils
from torchvision import transforms

from dataset_loaders.colour_mnist import (
    BG_NAMES,
    FG_NAMES,
    NUM_BG,
    NUM_DIGITS,
    NUM_FG,
    ColourMNIST,
)
from dataset_loaders.helpers import DATA_DIR
from utils.checkpoints import load_ae_from_path, load_nn_baseline_from_path
from utils.reproducibility import resolve_device, seed_everything
from utils.wandb_utils import load_from_wandb

device = resolve_device()
seed_everything(0)
print(device, "|", NUM_DIGITS * NUM_FG * NUM_BG, "combinations")

## Load the autoencoder and both baselines

In [ ]:
AE_ARTIFACT = "variational_colour_mnist_uniform"
BASELINE_ARTIFACTS = {
    "deterministic": "nn_baseline_colour_mnist_uniform_deterministic",
    "mixture": "nn_baseline_colour_mnist_uniform_mixture",
}

ae = load_ae_from_path(load_from_wandb(AE_ARTIFACT), device=device).to(device)
ae.eval()

baselines = {}
for name, artifact in BASELINE_ARTIFACTS.items():
    model = load_nn_baseline_from_path(load_from_wandb(artifact), device=device)
    baselines[name] = model.to(device).eval()
    print(
        f"{name:<14} num_vars={model.num_vars}  "
        f"components={getattr(model, 'num_components', 1)}"
    )

latent_dim = ae.get_latent_dim().numel()
for name, model in baselines.items():
    assert model.num_vars == latent_dim, (
        f"{name} expects {model.num_vars} latents but this AE has {latent_dim} -- "
        "the baseline was trained against a different autoencoder"
    )

## The combination grid

One row per digit. Within a row, three blocks of six: the background changes between
blocks, the foreground varies inside each block in `FG_NAMES` order.

In [ ]:
digit, fg, bg = torch.meshgrid(
    torch.arange(NUM_DIGITS),
    torch.arange(NUM_FG),
    torch.arange(NUM_BG),
    indexing="ij",
)
# permute to (digit, bg, fg) so a flattened row reads as bg-major, fg-minor.
all_labels = (
    torch.stack([digit, fg, bg], dim=-1).permute(0, 2, 1, 3).reshape(-1, 3).to(device)
)

ROW_WIDTH = NUM_FG * NUM_BG
print(f"{all_labels.shape[0]} labels -> {NUM_DIGITS} rows of {ROW_WIDTH}")
print("foregrounds:", ", ".join(FG_NAMES))
print("backgrounds:", ", ".join(BG_NAMES))

## Display helper

Two deliberate choices, both about being able to trust the colours:

- **no re-applied sigmoid.** `ae.decode` already returns values in `[0, 1]`; putting a
  second `torch.sigmoid` on top compresses everything toward 0.5 and washes the palette
  out.
- **`normalize=False`.** `make_grid`'s normalisation rescales by the grid's own min and
  max, so the same colour renders differently in two grids -- which defeats comparing
  one model against another.

In [ ]:
PADDING = 1


def show_grid(images, title, nrow=ROW_WIDTH, row_labels=None, block_labels=None):
    grid = vutils.make_grid(images.cpu(), nrow=nrow, normalize=False, padding=PADDING)
    height, width = grid.shape[1], grid.shape[2]

    plt.figure(figsize=(16, 16 * height / width), dpi=150)
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(title)

    cell = images.shape[-1] + PADDING
    if row_labels is not None:
        centres = [PADDING + i * cell + images.shape[-2] / 2 for i in range(len(row_labels))]
        plt.yticks(centres, row_labels, fontsize=7)
    else:
        plt.yticks([])

    if block_labels is not None:
        per_block = nrow // len(block_labels)
        centres = [
            PADDING + (i * per_block + per_block / 2) * cell for i in range(len(block_labels))
        ]
        plt.xticks(centres, block_labels, fontsize=7)
    else:
        plt.xticks([])

    for spine in plt.gca().spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    plt.show()


DIGIT_LABELS = [str(d) for d in range(NUM_DIGITS)]

## Reference: the real images

What the models are being asked to reproduce. Without this row the generated grids are
hard to read -- a colour that looks wrong may just be the palette.

In [ ]:
real_dataset = ColourMNIST(
    root=DATA_DIR, split="test", variant="uniform", transform=transforms.ToTensor()
)

targets = real_dataset.targets
picks = []
for row in all_labels.cpu():
    matches = (targets == row).all(dim=1).nonzero(as_tuple=True)[0]
    # The held-out variants have empty combinations; show black rather than crashing.
    picks.append(
        real_dataset[int(matches[0])][0] if len(matches) else torch.zeros(3, 28, 28)
    )
real_images = torch.stack(picks)

show_grid(
    real_images,
    "real colour-MNIST -- the target",
    row_labels=DIGIT_LABELS,
    block_labels=BG_NAMES,
)

## All 180 combinations, per baseline

In [ ]:
sampled = {}
with torch.no_grad():
    for name, model in baselines.items():
        sampled[name] = ae.decode(model.sample(all_labels)).cpu()

for name, images in sampled.items():
    show_grid(
        images,
        f"{name} -- all {images.shape[0]} combinations",
        row_labels=DIGIT_LABELS,
        block_labels=BG_NAMES,
    )

## What the grids above cannot show

Each grid is one draw per combination, so a point predictor and a distribution look
alike. Drawing the *same* combination repeatedly separates them: the deterministic
baseline's spread is exactly zero by construction.

In [ ]:
REPEATS = 12
PROBE = (3, 0, 1)  # digit 3, first foreground on the second background

probe_labels = torch.tensor([list(PROBE)] * REPEATS, device=device)
caption = f"digit {PROBE[0]} / {FG_NAMES[PROBE[1]]} on {BG_NAMES[PROBE[2]]}"

with torch.no_grad():
    for name, model in baselines.items():
        images = ae.decode(model.sample(probe_labels)).cpu()
        spread = images.std(dim=0).mean().item()
        show_grid(
            images,
            f"{name} -- {REPEATS} draws of {caption}  (std {spread:.5f})",
            nrow=REPEATS,
        )

## Trading diversity for fidelity

`std_correction` scales the noise the mixture adds after picking a component. Turning it
down makes samples cleaner and more alike, which is why models should be compared at
matched diversity rather than at a matched knob. The deterministic baseline ignores this
argument -- it has no noise to scale.

In [ ]:
first_row = all_labels[:ROW_WIDTH]

for std_correction in (0.25, 0.5, 1.0, 1.5):
    with torch.no_grad():
        images = ae.decode(
            baselines["mixture"].sample(first_row, std_correction=std_correction)
        ).cpu()
    show_grid(
        images,
        f"mixture -- digit 0, std_correction={std_correction}",
        block_labels=BG_NAMES,
    )